# Crypto Lab 09 — Building a Mental Model of TLS 1.3

## Today’s Big Question

**How does TLS combine authentication, key agreement, key derivation, and authenticated encryption into one secure channel?**

This notebook does **not** implement TLS. Real TLS is a large, carefully specified protocol and should be used through trusted libraries. Instead, we will model several of its cryptographic ideas with standard library primitives so that you can see where each operation belongs in the flow.

## Why This Matters

TLS is an example of a major course theme: **security is a system property**. None of the individual pieces below is “TLS” by itself.

A modern TLS connection needs to answer several different questions:

1. **Who am I talking to?** — certificates and signatures
2. **How do we establish fresh shared secrets?** — ephemeral Diffie-Hellman
3. **How do we turn one shared secret into several keys?** — HKDF
4. **How do we protect each application record?** — authenticated encryption
5. **How do we avoid nonce reuse and replay-like confusion?** — record sequence numbers and protocol state

The goal is to understand the responsibilities of each layer, not to reproduce the complete TLS specification.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from hashlib import sha256
import os

from cryptography.exceptions import InvalidTag
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ed25519, x25519
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

print("TLS modeling lab ready.")

TLS modeling lab ready.


## Part 1 — TLS Is More Than Encryption

Suppose a client performs Diffie-Hellman with *someone* on the network and then successfully encrypts traffic with the resulting key.

That gives confidentiality against passive observers, but it does **not** prove that the other endpoint is the intended server. An active attacker could perform one key exchange with the client and another with the real server.

TLS therefore combines **key agreement** with **authentication**.

### Exercise 1 — Separate the Security Goals

For each item below, identify the primary TLS mechanism that provides it.

| Goal | Mechanism                                                          |
|---|--------------------------------------------------------------------|
| Confidentiality of application records | **Your answer: Authenticated encryption like ChaCha20, AES, etc.** |
| Integrity/authenticity of application records | **Your answer: it has a signature feature in AES**                 |
| Server authentication | **Your answer: Certificats aka CA**                                |
| Fresh session secrets | **Your answer: every TLS session has an effemral DH key**          |
| Deriving several keys from one shared secret | **Your answer: it uses orderd nonces**                             |

Then answer: **Why would “just use AES” be an incomplete design for securing a web connection?**
- It might be an unsecure version, that version might not be available on the device connecting, cant verify the endpoint(server) and many other reasons.

## Part 2 — A Certificate as a Signed Binding

An X.509 certificate contains much more structure than we need for this demonstration. Conceptually, though, a certificate binds an identity (such as a DNS name) to a public key and is signed by an authority the client trusts.

The toy example below models that idea with Ed25519 signatures. This is **not** a replacement for certificate validation.

In [2]:
# Toy certificate authority and server identity.
ca_private = ed25519.Ed25519PrivateKey.generate()
ca_public = ca_private.public_key()

server_identity_private = ed25519.Ed25519PrivateKey.generate()
server_identity_public = server_identity_private.public_key()
server_pub_bytes = server_identity_public.public_bytes(
    encoding=serialization.Encoding.Raw,
    format=serialization.PublicFormat.Raw,
)

hostname = b"example.test"
certificate_body = hostname + b"|" + server_pub_bytes
certificate_signature = ca_private.sign(certificate_body)

# The client trusts ca_public and verifies the signed binding.
ca_public.verify(certificate_signature, certificate_body)
print("Toy certificate signature verified.")
print("Certified hostname:", hostname.decode())

Toy certificate signature verified.
Certified hostname: example.test


### Exercise 2 — Authentication Question

Assume an attacker can replace every packet between the client and server.

Why does verifying the certificate authority’s signature stop the attacker from simply substituting the attacker’s own server public key?

**Your answer:** It cannot replicate the same CA the sender is using, so we can verify the signature and authenticate the method.

## Part 3 — Ephemeral X25519 Key Agreement

TLS 1.3 normally uses an ephemeral Diffie-Hellman exchange. The long-term certificate key authenticates the handshake; it does not need to be the key that directly creates the shared traffic secret.

Here we use **X25519**, a widely deployed elliptic-curve Diffie-Hellman primitive.

In [3]:
client_dh_private = x25519.X25519PrivateKey.generate()
server_dh_private = x25519.X25519PrivateKey.generate()

client_dh_public = client_dh_private.public_key()
server_dh_public = server_dh_private.public_key()

client_shared = client_dh_private.exchange(server_dh_public)
server_shared = server_dh_private.exchange(client_dh_public)

print("Shared secrets match:", client_shared == server_shared)
print("Shared secret length:", len(client_shared), "bytes")
print("Shared secret fingerprint:", sha256(client_shared).hexdigest()[:24])

Shared secrets match: True
Shared secret length: 32 bytes
Shared secret fingerprint: dd092833d6ade15d791d2ada


### Exercise 3 — Forward Secrecy

Imagine the server deletes `server_dh_private` immediately after the connection is established but retains its long-term certificate signing key.

Later, an attacker steals the long-term signing key and has a packet capture of this old connection.

Why should the attacker still be unable to reconstruct the old Diffie-Hellman shared secret?

**Your answer:** since the old private key was deleted and they only have the public DH values, they cannot use it unless they break DH.

## Part 4 — HKDF: One Secret, Several Keys

A protocol should not use the raw Diffie-Hellman result as every key. TLS 1.3 uses an HKDF-based key schedule to derive separate secrets for separate purposes and directions.

We will derive:

- a client-to-server AES key,
- a server-to-client AES key,
- a client IV,
- a server IV.

Keeping directions separate prevents the same key/nonce pair from appearing in both directions.

In [4]:
def hkdf_expand(shared_secret: bytes, info: bytes, length: int) -> bytes:
    return HKDF(
        algorithm=hashes.SHA256(),
        length=length,
        salt=None,
        info=info,
    ).derive(shared_secret)

client_write_key = hkdf_expand(client_shared, b"classroom tls client write key", 16)
server_write_key = hkdf_expand(client_shared, b"classroom tls server write key", 16)
client_write_iv = hkdf_expand(client_shared, b"classroom tls client write iv", 12)
server_write_iv = hkdf_expand(client_shared, b"classroom tls server write iv", 12)

print("Client/server keys differ:", client_write_key != server_write_key)
print("Client/server IVs differ:", client_write_iv != server_write_iv)

Client/server keys differ: True
Client/server IVs differ: True


### Exercise 4 — Domain Separation

Why is it useful for the strings passed as `info` to be different when deriving client keys, server keys, and IVs?

**Your answer:** To stop repeat aka the values being the same when encrypted. This would allow an attacker to xor them reveling the secret used with them.

## Part 5 — Modeling TLS Record Nonces

TLS records do not send a fresh random nonce alongside every encrypted record. Instead, each endpoint keeps a sequence number and combines it with an IV derived during the handshake.

The exact TLS 1.3 construction has specification details we are intentionally omitting. The function below models the central idea: **the nonce changes with the record sequence number**.

In [5]:
def nonce_from_iv(base_iv: bytes, sequence_number: int) -> bytes:
    if len(base_iv) != 12:
        raise ValueError("This lab expects a 12-byte AEAD IV.")
    if not 0 <= sequence_number < 2**64:
        raise ValueError("Sequence number must fit in 64 bits.")

    seq_bytes = sequence_number.to_bytes(12, "big")
    return bytes(a ^ b for a, b in zip(base_iv, seq_bytes))

for seq in range(3):
    print(seq, nonce_from_iv(client_write_iv, seq).hex())

0 bd6890a4fa81d4c14ea761d2
1 bd6890a4fa81d4c14ea761d3
2 bd6890a4fa81d4c14ea761d0


### Exercise 5 — Finish a Nonce Check

Complete the assertion below so it confirms that the first 100 client record nonces are all unique.

In [6]:
client_nonces = [nonce_from_iv(client_write_iv, seq) for seq in range(100)]

# TODO: Replace the expression below with a uniqueness check.
nonce_check = len(client_nonces) == 100
print("100 nonce values generated:", nonce_check)
for i in client_nonces:
    for e in client_nonces:
        if i in client_nonces[i + 1]:
            return print("uniqueness: False")
print("uniqueness: True")
assert len(set(client_nonces)) == 100

100 nonce values generated: True


## Part 6 — Protecting a TLS-Like Record with AES-GCM

TLS 1.3 supports authenticated encryption. AES-GCM gives us both:

- ciphertext confidentiality, and
- an authentication tag that detects modification.

We will treat a small header as **associated data**: it is authenticated but not encrypted.

In [8]:
@dataclass
class ToyRecord:
    sequence_number: int
    associated_data: bytes
    ciphertext: bytes


def encrypt_record(key: bytes, base_iv: bytes, sequence_number: int,
                   plaintext: bytes, associated_data: bytes) -> ToyRecord:
    nonce = nonce_from_iv(base_iv, sequence_number)
    ciphertext = AESGCM(key).encrypt(nonce, plaintext, associated_data)
    return ToyRecord(sequence_number, associated_data, ciphertext)


def decrypt_record(key: bytes, base_iv: bytes, record: ToyRecord) -> bytes:
    nonce = nonce_from_iv(base_iv, record.sequence_number)
    return AESGCM(key).decrypt(nonce, record.ciphertext, record.associated_data)

record0 = encrypt_record(
    client_write_key,
    client_write_iv,
    sequence_number=0,
    plaintext=b"GET /grades HTTP/1.1",
    associated_data=b"content-type:application-data",
)

print("Ciphertext + tag:", record0.ciphertext.hex())
print("Decrypted:", decrypt_record(client_write_key, client_write_iv, record0))

Ciphertext + tag: 2d11e7b92bc98895216ca717c911a3ec2cfa277308423d6732baae4c2b94ab09ad4628d1
Decrypted: b'GET /grades HTTP/1.1'


### Exercise 6 — Tamper With the Record

Run the following cell. Explain why the receiver rejects the record rather than returning corrupted plaintext.

In [9]:
tampered = bytearray(record0.ciphertext)
tampered[0] ^= 0x01
bad_record = ToyRecord(record0.sequence_number, record0.associated_data, bytes(tampered))

try:
    decrypt_record(client_write_key, client_write_iv, bad_record)
    print("Unexpected: tampering was not detected")
except InvalidTag:
    print("Tampering detected: authentication tag verification failed")

Tampering detected: authentication tag verification failed


**Your explanation:** It detects it was decrypted wrong, we can see why above with tampered[0] ^= 0x01 this rejection stops potential tampering with the message or a MITM attack.

## Part 7 — Why Reusing a Record Sequence Number Is Dangerous

AEAD security depends on never reusing the same nonce with the same key. In our model, reusing a sequence number recreates the same nonce.

The protocol therefore needs state, not just encryption algorithms.

In [10]:
nonce_a = nonce_from_iv(client_write_iv, 7)
nonce_b = nonce_from_iv(client_write_iv, 7)
print("Same sequence number produces same nonce:", nonce_a == nonce_b)

nonce_c = nonce_from_iv(server_write_iv, 7)
print("Same sequence number in opposite direction produces a different nonce:", nonce_a != nonce_c)

Same sequence number produces same nonce: True
Same sequence number in opposite direction produces a different nonce: True


### Exercise 7 — Protocol State

Suppose a buggy implementation resets the client record sequence number to zero while continuing to use the same traffic key and base IV.

1. What cryptographic requirement would be violated?
2. Why is “AES-GCM is secure” not enough to save the application?

**Your answer:** this would allow for repeat use of Nonce and allow potential repeat attacks. AES-GCM like any alg depends on correct implementation not just the fact that the alg is secure. In this case secure implementation realize on good initial values.

## Part 8 — What TLS Can and Cannot Protect

For each scenario, decide whether the core TLS protocol should prevent the attack, or whether the attacker has moved outside TLS’s threat model.

| Scenario | TLS should stop it?       | Why?                                                         |
|---|---------------------------|--------------------------------------------------------------|
| Passive network eavesdropper reads application plaintext | Yes  |  TLS should protect it                                                            |
| Active attacker modifies an encrypted application record | Yes                       | It detects tampering                                         |
| Attacker presents an untrusted certificate for the wrong domain | Yes                       | Yes it verifies with the CA                                  |
| Malware fully controls the client after decryption | No                        | It happened after TLS was used                               |
| Attacker exploits SQL injection through a legitimate HTTPS request | no                        | in its eyes it is legitimate traffic TLS doesnt monitor that |
| Server process is compromised and reads plaintext before encryption | no                        | this also happened before TLS was used                       |

### Exercise 8

Fill in the table and identify the **boundary** where TLS stops being able to protect the system.

## Part 9 — Heartbleed and the Engineering Lesson

Heartbleed did not require an attacker to defeat AES, Diffie-Hellman, signatures, or certificate validation. It exploited an implementation bug that could expose memory beyond the intended buffer.

That makes it an especially important case study for this course:

> **Secure algorithms do not guarantee a secure system.**

### Reflection

How could a memory-safety bug expose cryptographic secrets even when every cryptographic algorithm is correctly chosen?

**Your answer:** The secrets or keys used to secure it needs to be stored somewhere. If the place the secret is being stored is broken into then it is almost impossible for a alg to secure it.

## Final Reflection

Answer each in 2–4 sentences.

1. Which TLS component would you remove if you wanted to make a connection vulnerable to a man-in-the-middle attack, and why?
- Remove the CA. That part proves who is sending it acting as TLS's signature.
2. What is the difference between the server’s long-term certificate key and its ephemeral Diffie-Hellman key?
- The DH key is used to establish the connection and keep forward security up, while the server certificate is used to prove the servers identity.
3. Why are separate traffic keys and sequence numbers part of the security design rather than mere implementation details?
- If an attacker can get the traffic keys and sequence numbers then they could break the user encryption. So to provide the best security they treat them as a security design rather than the users problem.
4. Give one example where TLS can be completely correct but the overall application is still insecure.
- There are 3 examples above. Malware fully controls the client after decryption, Attacker exploits SQL injection through a legitimate HTTPS request, Server process is compromised and reads plaintext before encryption. But I think the primary scenario is using TLS with a malicious actor, since it cannot monitor what they send you, and a insecure web app.
5. Finish the sentence: **TLS is not an encryption algorithm; it is…**
- a security framework

## Submission Checklist

- [ ] Completed Exercises 1–8
- [ ] Ran all provided demonstration cells
- [ ] Explained the tamper-detection result
- [ ] Completed the final reflection
- [ ] Can explain why this notebook is a **model of TLS concepts**, not a TLS implementation